# 30-day hospital readmissions by age

**Question** (from a hospital manager): *"What share of patients are readmitted within 30 days, and how does that differ by age group?"*

**Data:** the UCI dataset *Diabetes 130-US hospitals for years 1999–2008*: hospital stays of patients with diabetes in 130 US hospitals. Source, license and fingerprints are in [`data/README.md`](../data/README.md); medical terms are explained in [`docs/GLOSSARY.md`](../docs/GLOSSARY.md).

A *30-day readmission* means the patient was admitted to hospital again within 30 days of leaving.

## 1. Load the data

In `diabetic_data.csv`, missing values are written as the text `?`. pandas' default settings would also read the text `None` as missing, but here `None` is a real answer: "test not done" in `max_glu_serum` and `A1Cresult`. So only `?` is read as missing.

`low_memory=False` makes pandas read the whole file before choosing each column's type. Otherwise it warns that `payer_code` has mixed types, because its first 20,445 rows are all missing.

In [1]:
import pandas as pd

DATA = "../data/raw/"
df = pd.read_csv(DATA + "diabetic_data.csv", na_values="?", keep_default_na=False, low_memory=False)
df.shape

(101766, 50)

Check: `None` is kept as a real value, not turned into missing.

In [2]:
df["A1Cresult"].value_counts(dropna=False)

A1Cresult
None    84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64

## 2. Data-quality problems, before changing anything

Brief task 1. Every check below only reads `df`; nothing is changed. The list of problems is at the end of this section. It is built by code, so every count in it is computed.

### 2.1 Size and column types

In [3]:
print(f"{df.shape[0]:,} rows × {df.shape[1]} columns")
df.dtypes

101,766 rows × 50 columns


encounter_id                int64
patient_nbr                 int64
race                          str
gender                        str
age                           str
weight                        str
admission_type_id           int64
discharge_disposition_id    int64
admission_source_id         int64
time_in_hospital            int64
payer_code                    str
medical_specialty             str
num_lab_procedures          int64
num_procedures              int64
num_medications             int64
number_outpatient           int64
number_emergency            int64
number_inpatient            int64
diag_1                        str
diag_2                        str
diag_3                        str
number_diagnoses            int64
max_glu_serum                 str
A1Cresult                     str
metformin                     str
repaglinide                   str
nateglinide                   str
chlorpropamide                str
glimepiride                   str
acetohexamide 

There is no date column, so neither the order of a patient's stays nor the 30-day window can be checked.

### 2.2 Missing values (`?`)

In [4]:
COLUMNS = ["encounter_id", "patient_nbr", "age", "gender", "discharge_disposition_id", "readmitted"]

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing values in the six analysis columns:", df[COLUMNS].isna().sum().sum())
pd.DataFrame({"missing": missing, "percent": (100 * missing / len(df)).round(1)})

Missing values in the six analysis columns: 0


,missing,percent
weight,98569,96.9
medical_specialty,49949,49.1
payer_code,40256,39.6
race,2273,2.2
diag_3,1423,1.4
diag_2,358,0.4
diag_1,21,0.0


### 2.3 Stays and patients

In [5]:
stays_per_patient = df["patient_nbr"].value_counts()
repeat = stays_per_patient > 1
print("One row per stay (encounter_id is unique):", df["encounter_id"].is_unique)
print(f"Patients: {len(stays_per_patient):,}")
print(f"Patients with 2 or more stays: {repeat.sum():,} ({repeat.mean():.1%}), holding {stays_per_patient[repeat].sum():,} stays")
print("Most stays for one patient:", stays_per_patient.max())

One row per stay (encounter_id is unique): True
Patients: 71,518
Patients with 2 or more stays: 16,773 (23.5%), holding 47,021 stays
Most stays for one patient: 40


### 2.4 Allowed values in `age`, `gender` and `readmitted`

`value_counts(dropna=False)` shows every value, missing included. `isin()` then flags anything outside the allowed list; it is `False` for a missing value, so missing values are flagged too.

In [6]:
allowed = {
    "age": [f"[{a}-{a + 10})" for a in range(0, 100, 10)],
    "gender": ["Female", "Male"],
    "readmitted": ["NO", ">30", "<30"],
}
for col, values in allowed.items():
    print(df[col].value_counts(dropna=False).sort_index().to_string())
    unexpected = df.loc[~df[col].isin(values), col].value_counts(dropna=False)
    print(f"Missing or unexpected in {col}:", unexpected.to_dict() or "none", "\n")

age
[0-10)        161
[10-20)       691
[20-30)      1657
[30-40)      3775
[40-50)      9685
[50-60)     17256
[60-70)     22483
[70-80)     26068
[80-90)     17197
[90-100)     2793
Missing or unexpected in age: none 

gender
Female             54708
Male               47055
Unknown/Invalid        3
Missing or unexpected in gender: {'Unknown/Invalid': 3} 

readmitted
<30    11357
>30    35545
NO     54864
Missing or unexpected in readmitted: none 



### 2.5 Discharge codes and what they mean

The *discharge disposition* is where the patient went when leaving hospital: home, a care home, another hospital, hospice, or the patient died ("expired"). *Hospice* is comfort care for people expected to live only a few months.

The meanings come from the discharge table in `IDS_mapping.csv`, which holds three code tables in one file; `skiprows=10, nrows=30` reads only the discharge table. `keep_default_na=False` keeps code 18's label `NULL` visible.

In [7]:
pd.set_option("display.max_colwidth", None)  # show full labels
discharge = pd.read_csv(DATA + "IDS_mapping.csv", skiprows=10, nrows=30, keep_default_na=False)
discharge = discharge.set_index("discharge_disposition_id").sort_index()
discharge["stays"] = df["discharge_disposition_id"].value_counts().reindex(discharge.index, fill_value=0)
print("Codes in the data but not in the mapping file:", set(df["discharge_disposition_id"]) - set(discharge.index) or "none")
discharge

Codes in the data but not in the mapping file: none


,description,stays
discharge_disposition_id,,
1,Discharged to home,60234
2,Discharged/transferred to another short term hospital,2128
3,Discharged/transferred to SNF,13954
4,Discharged/transferred to ICF,815
5,Discharged/transferred to another type of inpatient care institution,1184
6,Discharged/transferred to home with home health service,12902
7,Left AMA,623
8,Discharged/transferred to home under care of Home IV provider,108
9,Admitted as an inpatient to this hospital,21


The codes that matter, looked up by their labels instead of typed by hand: death (labels starting with "Expired"), hospice (starting with "Hospice") and unknown ("NULL", "Not Mapped", "Unknown/Invalid").

In [8]:
label = discharge["description"]
DEATH = discharge.index[label.str.startswith("Expired")].tolist()
HOSPICE = discharge.index[label.str.startswith("Hospice")].tolist()
UNKNOWN = discharge.index[label.isin(["NULL", "Not Mapped", "Unknown/Invalid"])].tolist()
discharge.loc[DEATH + HOSPICE + UNKNOWN]

,description,stays
discharge_disposition_id,,
11,Expired,1642
19,"Expired at home. Medicaid only, hospice.",8
20,"Expired in a medical facility. Medicaid only, hospice.",2
21,"Expired, place unknown. Medicaid only, hospice.",0
13,Hospice / home,399
14,Hospice / medical facility,372
18,NULL,3691
25,Not Mapped,989
26,Unknown/Invalid,0


### 2.6 Can those patients come back?

In [9]:
outcome = pd.crosstab(df["discharge_disposition_id"], df["readmitted"])
outcome.reindex(DEATH + HOSPICE + UNKNOWN, fill_value=0)

readmitted,<30,>30,NO
discharge_disposition_id,,,
11,0,0,1642
19,0,0,8
20,0,0,2
21,0,0,0
13,19,36,344
14,24,7,341
18,459,1027,2205
25,92,376,521
26,0,0,0


### 2.7 The list of problems

In [10]:
discharge_code = df["discharge_disposition_id"]
death, hospice, unknown = discharge_code.isin(DEATH), discharge_code.isin(HOSPICE), discharge_code.isin(UNKNOWN)
death_no = (df.loc[death, "readmitted"] == "NO").sum()
hospice_back = (df.loc[hospice, "readmitted"] != "NO").sum()
hospice_30 = (df.loc[hospice, "readmitted"] == "<30").sum()
none_value = (df["max_glu_serum"] == "None") | (df["A1Cresult"] == "None")

problems = pd.DataFrame(
    [
        (f'Missing values are written as "?", in {len(missing)} columns (weight: {missing["weight"] / len(df):.0%} missing)',
         df.isna().any(axis=1).sum(),
         f"No: {df[COLUMNS].isna().sum().sum()} missing values in the six analysis columns"),
        ('By default, pandas would read the real value "None" (test not done) as missing, in max_glu_serum and A1Cresult',
         none_value.sum(),
         "No: avoided by loading with keep_default_na=False; these columns are not used"),
        (f"One patient can have many stays: {repeat.sum():,} of {len(stays_per_patient):,} patients have 2 or more",
         df["patient_nbr"].isin(stays_per_patient[repeat].index).sum(),
         "Yes: the table counts stays, not patients"),
        ('gender is "Unknown/Invalid"',
         (~df["gender"].isin(allowed["gender"])).sum(),
         "No: the age table does not use gender"),
        (f"The stay ended in death (codes {DEATH})",
         death.sum(),
         f'Yes: these patients cannot be readmitted ({death_no:,} of {death.sum():,} are flagged "NO")'),
        (f"The stay ended in hospice care (codes {HOSPICE})",
         hospice.sum(),
         f"Yes: not expected to return, yet {hospice_back} were readmitted, {hospice_30} of them within 30 days"),
        (f"The discharge destination is unknown (codes {UNKNOWN})",
         unknown.sum(),
         "Yes: these could hide deaths"),
        ("No dates, so the order of stays and the 30-day window cannot be checked",
         len(df),
         "Yes: the readmitted flag has to be trusted"),
    ],
    columns=["problem", "rows affected", "affects the question?"],
)
problems.index += 1
problems["rows affected"] = problems["rows affected"].map("{:,}".format)
problems

,problem,rows affected,affects the question?
1,"Missing values are written as ""?"", in 7 columns (weight: 97% missing)","100,723",No: 0 missing values in the six analysis columns
2,"By default, pandas would read the real value ""None"" (test not done) as missing, in max_glu_serum and A1Cresult","101,468",No: avoided by loading with keep_default_na=False; these columns are not used
3,"One patient can have many stays: 16,773 of 71,518 patients have 2 or more","47,021","Yes: the table counts stays, not patients"
4,"gender is ""Unknown/Invalid""",3,No: the age table does not use gender
5,"The stay ended in death (codes [11, 19, 20, 21])","1,652","Yes: these patients cannot be readmitted (1,652 of 1,652 are flagged ""NO"")"
6,"The stay ended in hospice care (codes [13, 14])",771,"Yes: not expected to return, yet 86 were readmitted, 43 of them within 30 days"
7,"The discharge destination is unknown (codes [18, 25, 26])","4,680",Yes: these could hide deaths
8,"No dates, so the order of stays and the 30-day window cannot be checked","101,766",Yes: the readmitted flag has to be trusted


Checked and fine: `encounter_id` is unique, `age` holds exactly the ten brackets, `readmitted` holds only `NO`, `>30` and `<30`, and every discharge code in the data appears in the mapping file.

Nothing in `df` was changed in this section. The exclusions come later, in section 3.

## 3. 30-day readmission rate by age (brief task 2)

Stays that ended in death or hospice care are excluded, because those patients cannot come back, or are not expected to. Their codes come from section 2.5. Each remaining stay counts once. Rate = stays flagged `<30` ÷ remaining stays.

In [11]:
excluded = df["discharge_disposition_id"].isin(DEATH + HOSPICE)
eligible = df[~excluded]
print(f"Excluded: {excluded.sum():,} stays (death or hospice). Remaining: {len(eligible):,}.")

Excluded: 2,423 stays (death or hospice). Remaining: 99,343.


In [12]:
import sys
sys.path.append("../src")  # where readmission.py lives
from readmission import readmission_rate_by_age

rates = readmission_rate_by_age(eligible)
rates.assign(readmission_rate_30d=rates["readmission_rate_30d"].map("{:.1%}".format))

,age_bracket,encounters,readmission_rate_30d
0,[0-10),160,1.9%
1,[10-20),690,5.8%
2,[20-30),1649,14.3%
3,[30-40),3764,11.3%
4,[40-50),9607,10.7%
5,[50-60),17060,9.8%
6,[60-70),22059,11.3%
7,[70-80),25331,12.1%
8,[80-90),16434,12.6%
9,[90-100),2589,11.9%


In [13]:
print(f"Overall 30-day readmission rate: {(eligible['readmitted'] == '<30').mean():.1%} of {len(eligible):,} stays")

Overall 30-day readmission rate: 11.4% of 99,343 stays
